In [1]:
%env ASTRA_DB_SECURE_BUNDLE_PATH=C:/Users/Mirosh/Documents/masterFp/apuntes iniciales python/proyectos/proyecto4/src/secure-connect-proyecto-4.zip
%env ASTRA_DB_APPLICATION_TOKEN=C:/Users/Mirosh/Documents/masterFp/apuntes iniciales python/proyectos/proyecto4/src/proyecto_4-token.json


env: ASTRA_DB_SECURE_BUNDLE_PATH=C:/Users/Mirosh/Documents/masterFp/apuntes iniciales python/proyectos/proyecto4/src/secure-connect-proyecto-4.zip
env: ASTRA_DB_APPLICATION_TOKEN=C:/Users/Mirosh/Documents/masterFp/apuntes iniciales python/proyectos/proyecto4/src/proyecto_4-token.json


In [2]:
from cassandra.cluster import Cluster
from cassandra.auth import PlainTextAuthProvider
import json
import os

In [3]:
cloud_config = {
    'secure_connect_bundle': os.environ['ASTRA_DB_SECURE_BUNDLE_PATH']
}

with open(os.environ['ASTRA_DB_APPLICATION_TOKEN']) as f:
    secrets = json.load(f)

CLIENT_ID = secrets['clientId']
CLIENT_SECRET = secrets['secret']
auth_provider = PlainTextAuthProvider(CLIENT_ID, CLIENT_SECRET)
cluster = Cluster(cloud=cloud_config, auth_provider=auth_provider)
session = cluster.connect()
session.set_keyspace('series')

In [4]:
keyspace_name = 'series'
query = (f"""CREATE TABLE IF NOT EXIST {keyspace_name}.stream
            show_id int,
            genre_name text,
            embeddings blob,
            embeddings_con_genero blob,
            vote_count int,
            name text,
            overview text,
            vote_average float,
            PRIMARY KEY ((show_id, genre_name, embeddings, embeddings_con_genero, vote_count))
        ); """)

session.execute(query)

SyntaxException: <Error from server: code=2000 [Syntax error in CQL query] message="line 1:20 mismatched input 'EXIST' expecting K_EXISTS (CREATE TABLE IF NOT [EXIST]...)">

In [7]:
import pandas as pd

shows_df = pd.read_csv('..\\datasets\\dataset_con_embbedings.csv')
shows_df

,show_id,name,overview,genre_name,embeddings,embeddings_con_genero,vote_count,vote_average
0,1399,Game of Thrones,Seven noble families fight for control of the ...,Sci-Fi & Fantasy/Drama/Action & Adventure,[-1.18278209e-02 1.22713363e-02 3.75224976e-...,"[0.0024598923046141863, -0.021662428975105286,...",21857,8.442
1,71446,Money Heist,To carry out the biggest heist in history a m...,Crime/Drama,[-9.99436621e-03 1.07302442e-01 -7.97618181e-...,"[-0.010140787810087204, 0.08295083791017532, -...",17836,8.257
2,66732,Stranger Things,When a young boy vanishes a small town uncove...,Drama/Sci-Fi & Fantasy/Mystery,[-4.96899709e-02 8.41833726e-02 -1.86806843e-...,"[-0.033907607197761536, 0.040717437863349915, ...",16161,8.624
3,1402,The Walking Dead,Sheriff's deputy Rick Grimes awakens from a co...,Action & Adventure/Drama/Sci-Fi & Fantasy,[-5.46765961e-02 -3.22576091e-02 -1.11697644e-...,"[-0.06431317329406738, -0.04335841163992882, -...",15432,8.121
4,63174,Lucifer,Bored and unhappy as the Lord of Hell Lucifer...,Crime/Sci-Fi & Fantasy,[ 4.13128510e-02 3.31981331e-02 -3.13385911e-...,"[0.011186204850673676, 0.02309972606599331, -0...",13870,8.486
...,...,...,...,...,...,...,...,...
16496,63878,Cradle to Grave,It's 1974 and 15 year-old Danny is our guide t...,Comedy/Drama,[-3.28636132e-02 7.44440928e-02 7.19452649e-...,"[-0.04765692725777626, 0.07140173763036728, -0...",6,8.500
16497,87569,Very Scary People,A chronicle of the twisted lives of some of th...,Documentary,[-1.00200512e-01 -9.53173265e-03 -4.30719070e-...,"[-0.0823121964931488, 0.000534385209903121, -0...",6,8.300
16498,100657,True Terror with Robert Englund,Horror movie icon Robert Englund journeys into...,NaN,[ 1.08446460e-02 -5.29182106e-02 -8.30574036e-...,"[0.010844646021723747, -0.0529182106256485, -0...",6,7.833
16499,50602,Dress to Kill,Dress To Kill is the title of a performance by...,NaN,[ 1.10674584e-02 3.06488667e-02 -5.27553186e-...,"[0.011067458428442478, 0.030648866668343544, -...",6,8.000


In [ ]:
import numpy as np

def list_to_bytes(l):
    arr = np.array(l, dtype=np.float32)  # o el tipo que sea correcto
    return arr.tobytes()

insert_query = """
INSERT INTO nombre_tabla (show_id, name, overview, genre_name, embeddings, embeddings_con_genero, vote_count, vote_average)
VALUES (?, ?, ?, ?, ?, ?, ?, ?)
"""

for idx, row in shows_df.iterrows():
    emb = eval(row['embeddings']) if isinstance(row['embeddings'], str) else row['embeddings']
    emb_bytes = list_to_bytes(emb)

    emb_gen = eval(row['embeddings_con_genero']) if isinstance(row['embeddings_con_genero'], str) else row['embeddings_con_genero']
    emb_gen_bytes = list_to_bytes(emb_gen)

    session.execute(insert_query, (
        row['show_id'],      # Asegúrate que es UUID o int según tu esquema
        row['name'],
        row['overview'],
        row['genre_name'],
        emb_bytes,
        emb_gen_bytes,
        int(row['vote_count']),
        float(row['vote_average'])
    ))


In [9]:
query = "SELECT * FROM series.stream limit 10"
rows = session.execute(query, timeout=60)
for row in rows:
    print(row)

Row(show_id=UUID('4157278a-33ec-58d1-847c-37304ac2e2bd'), genre_name='Crime/Drama', vote_count=6, embeddings=[-0.05815630778670311, 0.048798270523548126, -0.1266002058982849, -0.03756141662597656, 0.008134635165333748, 0.03608638420701027, 0.0719211995601654, 0.004302502144128084, -0.012501537799835205, 0.07738330960273743, 0.07493288069963455, 0.09561709314584732, 0.03514542058110237, -0.010499152354896069, -0.013788056559860706, -0.04317579045891762, 0.04629204422235489, -0.013029239140450954, -0.019914675503969193, -0.03473615646362305, -0.05212700739502907, -0.11543936282396317, 0.05830206349492073, -0.016337260603904724, -0.04662873223423958, 0.01568024791777134, 0.01697268895804882, -0.0402403362095356, -0.033656299114227295, -0.05903163552284241, 0.04386059194803238, -0.0619647353887558, 0.08133519440889359, 0.0582609623670578, 0.04102158918976784, 0.05189698562026024, 0.0719280019402504, -0.0654279887676239, 0.01633281074464321, 0.03637044504284859, 0.008421562612056732, -0.071